# Compose a custom anchored model from an expert package

**Feature:** NB-005  
**Goal:** install, explicitly synchronize, unload, and reinsert one validated package through the public `AnchoredMoEForCausalLM` API.

This notebook is a deterministic offline smoke workflow. The synthetic package setup stands in for the output of NB-003 and NB-004; the composition cells are identical for a real `ExpertPackage.from_directory(...)` descriptor.

## Resolve configuration

One mapping declares the anchor, converted layers, package identity, input, and exact-equivalence tolerances.

In [ ]:
import platform
from pathlib import Path
from tempfile import TemporaryDirectory

import safetensors
import torch
import transformers
from transformers import Qwen3Config, Qwen3ForCausalLM

from anchored_moe import (
    EXPERT_PAYLOAD_BINDING_FORMAT_VERSION,
    AffineRouterHead,
    AnchoredMoEForCausalLM,
    AppendOnlyRouterHeads,
    ExpertPackage,
    ExpertPackageManifest,
    FullMLPExpert,
    RoutedMLP,
)
from anchored_moe.evaluation import assert_tensors_equivalent
from anchored_moe.registry import PACKAGE_FORMAT_VERSION
from anchored_moe.utils import sha256_file, sha256_json

EXPERIMENT = {
    "feature_id": "NB-005",
    "seed": 13,
    "base_model_id": "local-random-qwen3",
    "base_model_revision": "synthetic-seed-13",
    "converted_layers": [2, 3],
    "specialization": "code",
    "specialization_version": "1.0.0",
    "expert_id": "code:0",
    "input_ids": [[1, 7, 11, 3]],
    "atol": 0.0,
    "rtol": 0.0,
}
EXPERIMENT

## Load and convert the immutable anchor

The dense reference is captured first. `from_anchor_model` then freezes the complete model and wraps only the configured MLPs.

In [ ]:
torch.manual_seed(EXPERIMENT["seed"])
base_config = Qwen3Config(
    vocab_size=64,
    hidden_size=16,
    intermediate_size=32,
    num_hidden_layers=4,
    num_attention_heads=4,
    num_key_value_heads=2,
    head_dim=4,
    max_position_embeddings=32,
)
anchor = Qwen3ForCausalLM(base_config).eval()
input_ids = torch.tensor(EXPERIMENT["input_ids"])
with torch.no_grad():
    dense_logits = anchor(input_ids=input_ids).logits

model = AnchoredMoEForCausalLM.from_anchor_model(
    anchor,
    converted_layers=EXPERIMENT["converted_layers"],
).eval()
assert model.conversion_report.converted_layers == tuple(EXPERIMENT["converted_layers"])
assert all(not parameter.requires_grad for parameter in model.parameters())

## Prepare a validated smoke package

This cell creates only the deterministic local fixture needed to execute the notebook offline. A normal workflow replaces the cell with `package = ExpertPackage.from_directory(package_path)` after NB-003 and NB-004.

In [ ]:
temporary_package = TemporaryDirectory(prefix="anchored-moe-nb005-")
package_root = Path(temporary_package.name)
weight_files = []
payload_bindings = []
package_hashes = {}
for layer_index in EXPERIMENT["converted_layers"]:
    wrapper = model.anchor_model.model.layers[layer_index].mlp
    assert isinstance(wrapper, RoutedMLP)
    expert = FullMLPExpert.from_anchor(wrapper.anchor_mlp, trainable=False)
    with torch.no_grad():
        for parameter in expert.parameters():
            parameter.zero_()
    relative_path = f"experts/layer-{layer_index}-code-0.safetensors"
    payload_path = package_root / relative_path
    payload_path.parent.mkdir(exist_ok=True)
    expert.save_safetensors(payload_path)
    weight_files.append(relative_path)
    payload_bindings.append({
        "layer_index": layer_index,
        "expert_id": EXPERIMENT["expert_id"],
        "weight_file": relative_path,
    })
    package_hashes[relative_path] = sha256_file(payload_path)

router_heads = AppendOnlyRouterHeads(base_config.hidden_size)
for layer_index in EXPERIMENT["converted_layers"]:
    head = AffineRouterHead(
        base_config.hidden_size,
        layer_index=layer_index,
        expert_id=EXPERIMENT["expert_id"],
        trainable=False,
    )
    with torch.no_grad():
        head.linear.bias.fill_(1.0)
    router_heads.append(head)
router_relative = "routers/affine-heads.safetensors"
router_path = package_root / router_relative
router_path.parent.mkdir()
router_heads.save_safetensors(router_path)
package_hashes[router_relative] = sha256_file(router_path)

manifest = ExpertPackageManifest(
    package_format_version=PACKAGE_FORMAT_VERSION,
    specialization_name=EXPERIMENT["specialization"],
    specialization_version=EXPERIMENT["specialization_version"],
    package_structure="single_expert",
    base_model_repository=EXPERIMENT["base_model_id"],
    base_model_revision=EXPERIMENT["base_model_revision"],
    base_config_sha256=sha256_json(model.anchor_model.config.to_dict()),
    base_weight_sha256={"model.safetensors": "0" * 64},
    tokenizer_repository=EXPERIMENT["base_model_id"],
    tokenizer_revision=EXPERIMENT["base_model_revision"],
    base_model_dtype="float32",
    converted_layers=EXPERIMENT["converted_layers"],
    expert_parameterization="full_mlp",
    expert_parameterization_config={
        "initialization": "anchor_copy",
        "payload_binding_format_version": EXPERT_PAYLOAD_BINDING_FORMAT_VERSION,
        "payload_bindings": payload_bindings,
    },
    expert_count=1,
    weight_dtype="float32",
    expert_weight_files=weight_files,
    router_architecture="independent_affine_heads",
    routing_policies_supported=["anchored_top1"],
    router_feature_specification={
        "feature_type": "identity",
        "input": "pre_mlp_hidden_state",
        "output_size": base_config.hidden_size,
        "version": 1,
    },
    router_files=[router_relative],
    runtime_versions={
        "accelerate": "not-installed",
        "kernels": "torch-default-cpu",
        "peft": "not-installed",
        "python": platform.python_version(),
        "safetensors": safetensors.__version__,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
    },
    training_dataset={
        "dataset_id": "synthetic-local-fixture",
        "revision": "1",
        "split": "train",
        "filters": [],
    },
    package_hashes=package_hashes,
)
manifest.save_json(package_root / "manifest.json")
package = ExpertPackage.from_directory(package_root)

## Compose, remove, and reinsert

Installation and unloading change registry state only. Each route change is applied by a visibly separate `synchronize_registry()` call.

In [ ]:
model.install(package)
with torch.no_grad():
    installed_but_unsynchronized_logits = model(input_ids=input_ids).logits
assert_tensors_equivalent(
    installed_but_unsynchronized_logits,
    dense_logits,
    name="installed but unsynchronized logits",
    atol=EXPERIMENT["atol"],
    rtol=EXPERIMENT["rtol"],
)

installed_report = model.synchronize_registry()
with torch.no_grad():
    routed_logits = model(input_ids=input_ids).logits
assert not torch.equal(routed_logits, dense_logits)

model.unload(package.identity)
with torch.no_grad():
    unloaded_but_unsynchronized_logits = model(input_ids=input_ids).logits
assert_tensors_equivalent(
    unloaded_but_unsynchronized_logits,
    routed_logits,
    name="unloaded but unsynchronized logits",
    atol=EXPERIMENT["atol"],
    rtol=EXPERIMENT["rtol"],
)

removed_report = model.synchronize_registry()
with torch.no_grad():
    removed_logits = model(input_ids=input_ids).logits
assert_tensors_equivalent(
    removed_logits,
    dense_logits,
    name="removed forced-anchor logits",
    atol=EXPERIMENT["atol"],
    rtol=EXPERIMENT["rtol"],
)

model.install(ExpertPackage.from_directory(package_root))
reinserted_report = model.synchronize_registry()
with torch.no_grad():
    reinserted_logits = model(input_ids=input_ids).logits

## Validate lifecycle invariants

Reinsertion must restore the registry identity and routed logits exactly, while unloading plus synchronization must restore the dense anchor exactly.

In [ ]:
reinserted_result = assert_tensors_equivalent(
    reinserted_logits,
    routed_logits,
    name="reinserted routed logits",
    atol=EXPERIMENT["atol"],
    rtol=EXPERIMENT["rtol"],
)
assert reinserted_report.registry_sha256 == installed_report.registry_sha256
assert removed_report.reset_layers == tuple(EXPERIMENT["converted_layers"])
assert reinserted_result.exact
composition_summary = {
    "feature_id": EXPERIMENT["feature_id"],
    "converted_layers": installed_report.configured_layers,
    "expert_ids_by_layer": installed_report.expert_ids_by_layer,
    "install_required_explicit_sync": True,
    "unload_required_explicit_sync": True,
    "exact_anchor_restoration": True,
    "exact_reinsertion": reinserted_result.exact,
    "registry_sha256_restored": reinserted_report.registry_sha256 == installed_report.registry_sha256,
}
print("NB-005 validation passed", composition_summary)
temporary_package.cleanup()